> Projeto Desenvolve <br>
Programação Intermediária com Python <br>
Profa. Camila Laranjeira (mila@projetodesenvolve.com.br) <br>

# 4.2 - APIs


## Exercícios 🔭🌌🪐

Vamos acessar as APIs da NASA para ver algumas imagens interessantes capturadas universo afora!

#### Q1.
Crie uma chave no site oficial:
* https://api.nasa.gov

Vamos armazenar a chave de forma segura! <br>
Salve a sua chave em um arquivo `key.json` na forma:
`API_KEY=SUA_CHAVE`

Adicione o nome do arquivo `key.json` ao `.gitignore` do repositório que você fará upload da atividade.
Para isso basta abrir o arquivo `.gitignore` na pasta raíz do repositório (ou criar um caso ele não exista). Dentro do arquivo, apenas adiciona o nome do arquivo que deseja ignorar.

#### Q2. 🛰 Astronomy Picture of the Day (APOD) 🌌
> Antes de fazer os exercícios, devo te lembrar que existem limites de acesso às APIs, descritas na página principal, portanto pega leve na tentativa e erro na hora de testar seu código.

<img width=500 src=https://apod.nasa.gov/apod/image/2407/M24-HaLRGB-RC51_1024.jpg>

A primeira API que acessaremos é a mais popular de todas: astronomy picture of the day (foto astronômica do dia).

Faça uma requisição GET para a URL da API que retorna a imagem do dia! Essa é fácil já que são os valores padrão da rota principal:
* URL base: `'https://api.nasa.gov/planetary/apod'`
* Endpoint: não precisa preencher, acessaremos a raíz da API.
* Query params: preencha `api_key` com a sua chave de autenticação. Se animar mexer em outros parâmetros veja [a documentação](https://api.nasa.gov).

Ao receber a resposta (um json), você deve:
* Imprimir os campos `copyright` e `explanation`
* Com as biblioteca scikit-images e matplotlib, apresente a imagem a partir do campo `url` ou `hdurl`, e preencha o título do plot com o campo `title` do json. Uma dica de código a seguir.
```python
from skimage import io
img = io.imread(url)
## plot a matriz img com matplotlib (imshow)
```   

In [ ]:
# Q1 + Q2
# - l? API key
# - garante key.json no .gitignore
# - consulta APOD e plota imagem
import os
import requests
from skimage import io
import matplotlib.pyplot as plt

# garantir key.json no .gitignore
if os.path.exists(".gitignore"):
    with open(".gitignore", "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f.readlines()]
else:
    lines = []

if "key.json" not in lines:
    with open(".gitignore", "a", encoding="utf-8") as f:
        f.write("\nkey.json\n")

# ler chave
with open("key.json", "r", encoding="utf-8") as f:
    content = f.read().strip()

api_key = content.split("=", 1)[1].strip() if "=" in content else content

# APOD
url = "https://api.nasa.gov/planetary/apod"
resp = requests.get(url, params={"api_key": api_key})
resp.raise_for_status()

data = resp.json()
print(data.get("copyright"))
print(data.get("explanation"))

img_url = data.get("hdurl") or data.get("url")
img = io.imread(img_url)
plt.imshow(img)
plt.axis("off")
plt.title(data.get("title", "APOD"))
plt.show()


#### Q3. Limites
A partir da resposta da query anterios, imprima o header da resposta e consulte os atributos:
* X-RateLimit-Limit: o limite total de requisições da sua chave de API
* X-RateLimit-Remaining: o limite restante de requisições da sua chave de API

In [ ]:
# Q3 - Limites da API
print("X-RateLimit-Limit:", resp.headers.get("X-RateLimit-Limit"))
print("X-RateLimit-Remaining:", resp.headers.get("X-RateLimit-Remaining"))


### Q4. Mars Rover Photos 🚀🚙 📷

<img width=500 src=https://www.nasa.gov/wp-content/uploads/2019/10/pia23378-16.jpg>

Essa API retorna dados (incluindo imagens capturadas) sobre os veículos que hoje habitam o planeta Marte. São os rovers `opportunity`, `spirit` e o mais famoso, o `curiosity` (da foto acima).

Antes de requisitar imagens, vamos ver o relatório de dados coletados por um deles, o `curiosity`. Isso vai nos ajudar a montar a query de imagens coletadas.

Faça uma requisição GET para a seguinte URL:
* URL base: `'https://api.nasa.gov/mars-photos/api/v1'`
* endpoint: `'/manifests/{nome_do_rover}'`
* query parameters: preencha `api_key` com a sua chave de autenticação.

Extraia o json da resposta retornada. O campo principal é o `'photo_manifest'`, do qual queremos acessar os seguintes valores:
* `max_sol`: Máximo "dia marciano" de coleta de fotos. O dia marciano tem 24 horas, 39 minutos e 35 segundos.
* `max_date`: Última data terrestre de coleta de fotos, na forma `'aaaa-mm-dd'`.

Imprima esses dois atributos da resposta e os use no próximo exercício para coletar as fotos mais recentes tiradas. 

In [ ]:
# Q4 - Mars Rover Photos (manifest)
rover = "curiosity"
base_url = "https://api.nasa.gov/mars-photos/api/v1"

manifest_resp = requests.get(
    f"{base_url}/manifests/{rover}",
    params={"api_key": api_key}
)
manifest_resp.raise_for_status()
manifest = manifest_resp.json().get("photo_manifest", {})

max_sol = manifest.get("max_sol")
max_date = manifest.get("max_date")
print("max_sol:", max_sol)
print("max_date:", max_date)


#### Q5.

Faça uma requisição GET para a URL da API que retorna links para as imagens coletadas pelos rovers.

* URL base: `'https://api.nasa.gov/mars-photos/api/v1'`
* Endpoint: `/rovers/{nome_do_rover}/photos`
* Query params sugeridos: 
    * `api_key`: sua chave de autenticação.
    * `sol`: dia marciano que deseja coletar (de 0 a `max_sol` coletado anteriormente)
    * `page`: você pode paginar entre as respostas! São retornados 25 resultados por página.

A resposta esperada estará no formato a seguir, uma lista no campo `'photos'` onde cada item é um dicionário com os dados da foto tirada. Dentre os dados há o campo `camera` indicando qual das câmeras do rover tirou a foto. As fotos mais interessantes (na minha opinião, claro) são das câmeras de navegação (`"name": "NAVCAM"`) e as de prevenção de colisão (frente: `"name": "FHAZ"` e trás `"name": "RHAZ"`) onde dá pra ver partes do robô!

**Seu trabalho é**:
* Paginar a requisição acima até que a resposta seja `None`
* Escolher uma ou mais câmeras (ex: `NAVCAM`, `FHAZ`, `RHAZ`), e em um laço de repetição plotar todas as imagens retornadas daquela câmera. Use novamente as bibliotecas scikit-image e matplotlib. 
  * O título da imagem deve ter a página da requisição, nome da câmera e id da imagem.

```json
{
  "photos": [
    {
      "id": 1228212,
      "sol": 4102,
      "camera": {
        "id": 20,
        "name": "FHAZ",
        "rover_id": 5,
        "full_name": "Front Hazard Avoidance Camera"
      },
      "img_src": "https://mars.nasa.gov/msl-raw-images/proj/msl/redops/ods/surface/sol/04102/opgs/edr/fcam/FLB_761645828EDR_F1060660FHAZ00302M_.JPG",
      "earth_date": "2024-02-19",
      "rover": {
        "id": 5,
        "name": "Curiosity",
        ...
      }
    }
    {
      "id": 1228213,
      "sol": 4102, 
      ...
    }
```



In [ ]:
# Q5 - Mars Rover Photos (pagina??o e plot)
import matplotlib.pyplot as plt
from skimage import io

cameras = {"NAVCAM", "FHAZ", "RHAZ"}
page = 1

while True:
    photos_resp = requests.get(
        f"{base_url}/rovers/{rover}/photos",
        params={"api_key": api_key, "sol": max_sol, "page": page}
    )
    photos_resp.raise_for_status()
    photos = photos_resp.json().get("photos", [])

    if not photos:
        break

    for p in photos:
        cam_name = p.get("camera", {}).get("name")
        if cam_name in cameras:
            img_url = p.get("img_src")
            img = io.imread(img_url)
            plt.figure(figsize=(6,4))
            plt.imshow(img)
            plt.axis("off")
            plt.title(f"page {page} | {cam_name} | id {p.get('id')}")
            plt.show()

    page += 1
